# Kaggle Pipeline - Generate Mock User Interactions for SASRec

Notebook nay sinh `mock_user_profiles + user_interactions + sasrec sequences` tu bo `item_metadata_final.jsonl` da duoc QA va dedup.

## Input can co
- `outputs/item_metadata_final.jsonl` tu notebook tao clips
- Neu chua co ban final, co the fallback sang `item_metadata_after_qa.jsonl` hoac `item_metadata.jsonl`
- Hoac 1 Kaggle Dataset co chua cac file tren

## Output se tao ra
- `mock_outputs/item_id_map.json`
- `mock_outputs/mock_user_profiles.jsonl`
- `mock_outputs/user_interactions.jsonl`
- `mock_outputs/user_interactions.parquet`
- `mock_outputs/sasrec_train_sequences.jsonl`
- `mock_outputs/sasrec_valid_sequences.jsonl`
- `mock_outputs/sasrec_test_sequences.jsonl`
- `mock_outputs/interaction_summary.json`

## Muc tieu
- Tao log hanh vi du giong that de test pipeline train SASRec
- Giu schema gan voi log that de sau nay thay bang production data de hon
- Tao du sequence train/valid/test de test nhanh model

## Cach chay tren Kaggle
1. Attach dataset co `item_metadata_final.jsonl`, hoac dung file trong `/kaggle/working/outputs/` neu vua chay notebook clip xong.
2. Sua `ITEM_METADATA_PATH` trong cell config neu can.
3. Run notebook tu tren xuong duoi.
4. Download thu muc `mock_outputs/` sau khi chay xong.

## Ghi chu
- Day la du lieu gia lap, dung de test train va debug pipeline.
- Notebook mac dinh uu tien bo final da QA + dedup.
- Ban co the doi so luong user/session trong cell config de scale nhanh hoac lon hon.


In [ ]:
%pip install -q orjson pyarrow


In [ ]:
from pathlib import Path
import json
import random
import warnings
from typing import Dict, Any, List

import numpy as np
import pandas as pd
import orjson
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RNG = random.Random(SEED)

# Sua path nay theo dataset ban attach vao notebook.
DATASET_ROOT = Path("/kaggle/input/datasets/sonisson/ready-items")
WORK_DIR = Path("/kaggle/working")

ITEM_METADATA_PATH = Path("/kaggle/input/datasets/sonisson/ready-items/publish/ready_items_v1/item_metadata_final.jsonl")

OUTPUT_DIR = WORK_DIR / "mock_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Config sinh mock data
NUM_USERS = 1000
SESSION_DAY_SPAN = 30
MAX_SEQ_LEN = 20

print("DATASET_ROOT:", DATASET_ROOT)
print("ITEM_METADATA_PATH:", ITEM_METADATA_PATH)
print("ITEM_METADATA_PATH exists:", ITEM_METADATA_PATH.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)
print("NUM_USERS:", NUM_USERS)
print("SEED:", SEED)


In [ ]:
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "rb") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(orjson.loads(line))
    return rows

def write_jsonl(records: List[Dict[str, Any]], path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        for row in records:
            f.write(orjson.dumps(row))
            f.write(b"\n")

def write_json(data: Dict[str, Any], path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        f.write(orjson.dumps(data, option=orjson.OPT_INDENT_2))

def clean_text(text: Any) -> str:
    if text is None:
        return ""
    return " ".join(str(text).strip().split())

def safe_float(value: Any, default: float = 0.0) -> float:
    try:
        return float(value)
    except Exception:
        return default

def clamp(value: float, low: float, high: float) -> float:
    return max(low, min(high, value))

def normalize_score(value: float, low: float = 0.0, high: float = 10.0) -> float:
    if high <= low:
        return 0.0
    return clamp((value - low) / (high - low), 0.0, 1.0)

def sample_num_sessions(rng: random.Random) -> int:
    r = rng.random()
    if r < 0.65:
        return rng.randint(4, 9)
    if r < 0.92:
        return rng.randint(10, 20)
    return rng.randint(21, 40)

def sample_session_length(rng: random.Random) -> int:
    r = rng.random()
    if r < 0.20:
        return rng.randint(5, 7)
    if r < 0.80:
        return rng.randint(8, 15)
    return rng.randint(16, 25)

def pick_weighted_row(rows: List[Dict[str, Any]], rng: random.Random) -> Dict[str, Any]:
    if not rows:
        raise ValueError("rows must not be empty")
    weights = []
    for row in rows:
        quality = row.get("item_quality_score", 0.5)
        weights.append(max(0.05, 0.15 + quality))
    return rng.choices(rows, weights=weights, k=1)[0]

def build_profile(user_id: str, rng: random.Random, keyword_pool: List[str], host_pool: List[str]) -> Dict[str, Any]:
    num_keywords = rng.choices([1, 2, 3], weights=[0.35, 0.45, 0.20], k=1)[0]
    fav_keywords = rng.sample(keyword_pool, k=min(num_keywords, len(keyword_pool))) if keyword_pool else []
    num_hosts = rng.choices([1, 2], weights=[0.70, 0.30], k=1)[0]
    fav_hosts = rng.sample(host_pool, k=min(num_hosts, len(host_pool))) if host_pool else []
    return {
        "user_id": user_id,
        "fav_keywords": fav_keywords,
        "fav_hosts": fav_hosts,
        "patience_level": round(rng.uniform(0.15, 0.80), 3),
        "like_rate_bias": round(rng.uniform(0.00, 0.04), 3),
        "save_rate_bias": round(rng.uniform(0.00, 0.015), 3),
        "share_rate_bias": round(rng.uniform(0.00, 0.008), 3),
        "follow_rate_bias": round(rng.uniform(0.00, 0.015), 3),
        "explore_rate": round(rng.uniform(0.18, 0.45), 3)
    }

def sample_item_for_user(
    profile: Dict[str, Any],
    rng: random.Random,
    all_items: List[Dict[str, Any]],
    keyword_index: Dict[str, List[Dict[str, Any]]],
    host_index: Dict[str, List[Dict[str, Any]]],
    used_clip_ids: set,
) -> Dict[str, Any]:
    buckets = ["fav_keyword", "fav_host", "explore"]
    weights = [0.50, 0.20, 0.30]
    bucket = rng.choices(buckets, weights=weights, k=1)[0]

    candidates: List[Dict[str, Any]] = []
    if bucket == "fav_keyword" and profile.get("fav_keywords"):
        keyword = rng.choice(profile["fav_keywords"])
        candidates = [row for row in keyword_index.get(keyword, []) if row["clip_id"] not in used_clip_ids]
    elif bucket == "fav_host" and profile.get("fav_hosts"):
        host = rng.choice(profile["fav_hosts"])
        candidates = [row for row in host_index.get(host, []) if row["clip_id"] not in used_clip_ids]

    if not candidates:
        candidates = [row for row in all_items if row["clip_id"] not in used_clip_ids]
    if not candidates:
        candidates = all_items

    return pick_weighted_row(candidates, rng)

def simulate_event(
    row: Dict[str, Any],
    profile: Dict[str, Any],
    session_id: str,
    event_id: str,
    position_in_session: int,
    session_length: int,
    event_time: pd.Timestamp,
    rng: random.Random,
) -> Dict[str, Any]:
    duration = max(safe_float(row.get("duration_sec"), 30.0), 1.0)
    keyword_match = 1.0 if row.get("keyword_norm", "") in profile.get("fav_keywords", []) else 0.0
    host_match = 1.0 if row.get("host_norm", "") in profile.get("fav_hosts", []) else 0.0
    quality = row.get("item_quality_score", 0.5)
    complete_bonus = 1.0 if row.get("is_sentence_complete", False) else 0.0

    affinity = (
        0.28 * keyword_match
        + 0.14 * host_match
        + 0.17 * quality
        + 0.08 * complete_bonus
        + 0.07 * profile.get("patience_level", 0.5)
        + 0.04 * (1.0 - profile.get("explore_rate", 0.2))
        + rng.uniform(-0.25, 0.25)
    )
    affinity = clamp(affinity, 0.0, 1.0)

    if affinity >= 0.75:
        band = rng.choices(["skip_early", "shallow_watch", "good_watch", "deep_watch"], weights=[0.08, 0.24, 0.42, 0.26], k=1)[0]
    elif affinity >= 0.55:
        band = rng.choices(["skip_early", "shallow_watch", "good_watch", "deep_watch"], weights=[0.16, 0.32, 0.34, 0.18], k=1)[0]
    elif affinity >= 0.35:
        band = rng.choices(["skip_early", "shallow_watch", "good_watch", "deep_watch"], weights=[0.28, 0.38, 0.25, 0.09], k=1)[0]
    else:
        band = rng.choices(["skip_early", "shallow_watch", "good_watch", "deep_watch"], weights=[0.42, 0.36, 0.17, 0.05], k=1)[0]

    completion_ranges = {
        "skip_early": (0.01, 0.14),
        "shallow_watch": (0.14, 0.40),
        "good_watch": (0.40, 0.80),
        "deep_watch": (0.80, 1.08),
    }
    low, high = completion_ranges[band]
    completion_rate = round(rng.uniform(low, high), 3)
    watch_time_sec = round(duration * completion_rate, 2)

    is_skip = int(completion_rate < 0.16)
    is_finish = int(completion_rate >= 0.88)
    is_replay = int(completion_rate > 1.00)

    like_prob = clamp(0.005 + 0.10 * max(completion_rate - 0.68, 0) + 0.04 * keyword_match + 0.03 * quality + profile.get("like_rate_bias", 0.0), 0.0, 0.25)
    save_prob = clamp(0.001 + 0.04 * max(completion_rate - 0.82, 0) + 0.02 * quality + profile.get("save_rate_bias", 0.0), 0.0, 0.10)
    share_prob = clamp(0.0005 + 0.02 * max(completion_rate - 0.92, 0) + 0.01 * quality + profile.get("share_rate_bias", 0.0), 0.0, 0.05)
    follow_prob = clamp(0.0005 + 0.02 * host_match * max(completion_rate - 0.78, 0) + profile.get("follow_rate_bias", 0.0), 0.0, 0.06)

    is_like = int(rng.random() < like_prob)
    is_save = int(rng.random() < save_prob)
    is_share = int(rng.random() < share_prob)
    is_follow_host = int(rng.random() < follow_prob)

    engagement_score = (
        0.64 * min(completion_rate, 1.0)
        + 0.11 * is_like
        + 0.08 * is_save
        + 0.04 * is_share
        + 0.05 * is_finish
        + 0.04 * quality
        + 0.04 * keyword_match
    )
    engagement_score = round(clamp(engagement_score, 0.0, 1.0), 3)

    strong_positive = completion_rate >= 0.72 or (completion_rate >= 0.62 and (is_like or is_save or is_share))
    weak_positive = completion_rate >= 0.48 and (is_like or (keyword_match and quality >= 0.6 and completion_rate >= 0.52))
    watch_label = 1 if (strong_positive or weak_positive) else 0

    if watch_label == 1 and (completion_rate >= 0.88 or is_save or is_share):
        signal_strength = "strong_positive"
    elif watch_label == 1:
        signal_strength = "positive"
    elif completion_rate < 0.16:
        signal_strength = "weak_negative"
    else:
        signal_strength = "weak"

    source_surface = rng.choices(["home_feed", "related_feed", "search", "explore_feed"], weights=[0.62, 0.18, 0.05, 0.15], k=1)[0]

    return {
        "event_id": event_id,
        "user_id": profile["user_id"],
        "session_id": session_id,
        "event_time": event_time.isoformat().replace("+00:00", "Z"),
        "clip_id": row["clip_id"],
        "item_id": int(row["item_id"]),
        "episode_id": row.get("episode_id", ""),
        "position_in_session": position_in_session,
        "session_length": session_length,
        "source_surface": source_surface,
        "impression": 1,
        "play_start": 1,
        "watch_time_sec": watch_time_sec,
        "clip_duration_sec": round(duration, 2),
        "completion_rate": completion_rate,
        "is_skip": is_skip,
        "is_finish": is_finish,
        "is_replay": is_replay,
        "is_like": is_like,
        "is_save": is_save,
        "is_share": is_share,
        "is_follow_host": is_follow_host,
        "engagement_score": engagement_score,
        "watch_label": watch_label,
        "signal_strength": signal_strength,
    }


In [ ]:
if not ITEM_METADATA_PATH.exists():
    raise FileNotFoundError(f"Khong tim thay item_metadata.jsonl tai: {ITEM_METADATA_PATH}")

raw_items = read_jsonl(ITEM_METADATA_PATH)
prepared_items = []

for idx, row in enumerate(raw_items, start=1):
    clip_id = clean_text(row.get("clip_id"))
    if not clip_id:
        continue

    duration_sec = max(safe_float(row.get("duration_sec"), 30.0), 1.0)
    llm_score = safe_float(row.get("llm_score"), safe_float(row.get("heuristic_score"), 5.0))
    heuristic_score = safe_float(row.get("heuristic_score"), 5.0)
    sentence_complete = bool(row.get("is_sentence_complete", False))
    keyword_norm = clean_text(row.get("keyword")).lower()
    host_norm = clean_text(row.get("host")).lower()

    item_quality_score = (
        0.55 * normalize_score(llm_score, 0.0, 10.0)
        + 0.30 * normalize_score(heuristic_score, 0.0, 7.0)
        + 0.15 * (1.0 if sentence_complete else 0.0)
    )
    item_quality_score = round(clamp(item_quality_score, 0.0, 1.0), 3)

    prepared = dict(row)
    prepared["item_id"] = idx
    prepared["duration_sec"] = round(duration_sec, 2)
    prepared["keyword_norm"] = keyword_norm
    prepared["host_norm"] = host_norm
    prepared["item_quality_score"] = item_quality_score
    prepared_items.append(prepared)

if not prepared_items:
    raise ValueError("Khong co clip hop le trong item_metadata.jsonl")

catalog_df = pd.DataFrame(prepared_items)
keyword_index: Dict[str, List[Dict[str, Any]]] = {}
host_index: Dict[str, List[Dict[str, Any]]] = {}

for row in prepared_items:
    if row["keyword_norm"]:
        keyword_index.setdefault(row["keyword_norm"], []).append(row)
    if row["host_norm"]:
        host_index.setdefault(row["host_norm"], []).append(row)

keyword_pool = sorted([k for k, rows in keyword_index.items() if len(rows) >= 2])
host_pool = sorted([h for h, rows in host_index.items() if len(rows) >= 2])

item_id_map = {row["clip_id"]: int(row["item_id"]) for row in prepared_items}
item_id_map_path = OUTPUT_DIR / "item_id_map.json"
write_json(item_id_map, item_id_map_path)

print("items:", len(prepared_items))
print("unique_keywords:", len(keyword_pool))
print("unique_hosts:", len(host_pool))
print("saved:", item_id_map_path)
catalog_df[["clip_id", "item_id", "episode_id", "keyword", "host", "duration_sec", "item_quality_score"]].head(5)


In [ ]:
user_profiles = []

for user_idx in range(1, NUM_USERS + 1):
    user_id = f"user_{user_idx:05d}"
    profile = build_profile(user_id, RNG, keyword_pool, host_pool)
    user_profiles.append(profile)

profiles_path = OUTPUT_DIR / "mock_user_profiles.jsonl"
write_jsonl(user_profiles, profiles_path)

print("users:", len(user_profiles))
print("saved:", profiles_path)
pd.DataFrame(user_profiles).head(5)


In [ ]:
events = []
event_counter = 1
base_time = pd.Timestamp("2026-04-01T07:00:00Z")

for profile in tqdm(user_profiles, desc="generate_sessions"):
    num_sessions = sample_num_sessions(RNG)
    user_time = base_time + pd.Timedelta(days=RNG.randint(0, SESSION_DAY_SPAN - 1), minutes=RNG.randint(0, 16 * 60))

    for session_idx in range(1, num_sessions + 1):
        session_id = f"sess_{profile['user_id']}_{session_idx:04d}"
        session_length = sample_session_length(RNG)
        used_clip_ids = set()
        cursor_time = user_time

        for position in range(1, session_length + 1):
            item_row = sample_item_for_user(profile, RNG, prepared_items, keyword_index, host_index, used_clip_ids)
            used_clip_ids.add(item_row["clip_id"])

            event = simulate_event(
                row=item_row,
                profile=profile,
                session_id=session_id,
                event_id=f"evt_{event_counter:09d}",
                position_in_session=position,
                session_length=session_length,
                event_time=cursor_time,
                rng=RNG,
            )
            events.append(event)
            event_counter += 1

            gap_sec = max(3, int(min(event["watch_time_sec"], event["clip_duration_sec"]) + RNG.uniform(2, 25)))
            cursor_time = cursor_time + pd.Timedelta(seconds=gap_sec)

        user_time = cursor_time + pd.Timedelta(minutes=RNG.randint(20, 720))

interactions_df = pd.DataFrame(events)
interactions_df = interactions_df.sort_values(["user_id", "event_time", "position_in_session"]).reset_index(drop=True)

interactions_path_jsonl = OUTPUT_DIR / "user_interactions.jsonl"
interactions_path_parquet = OUTPUT_DIR / "user_interactions.parquet"
write_jsonl(interactions_df.to_dict(orient="records"), interactions_path_jsonl)
interactions_df.to_parquet(interactions_path_parquet, index=False)

summary = {
    "users": int(interactions_df["user_id"].nunique()),
    "sessions": int(interactions_df["session_id"].nunique()),
    "events": int(len(interactions_df)),
    "items": int(interactions_df["clip_id"].nunique()),
    "avg_session_length": round(float(interactions_df.groupby("session_id").size().mean()), 2),
    "avg_watch_time_sec": round(float(interactions_df["watch_time_sec"].mean()), 2),
    "avg_completion_rate": round(float(interactions_df["completion_rate"].mean()), 3),
    "skip_rate": round(float(interactions_df["is_skip"].mean()), 3),
    "finish_rate": round(float(interactions_df["is_finish"].mean()), 3),
    "like_rate": round(float(interactions_df["is_like"].mean()), 3),
    "save_rate": round(float(interactions_df["is_save"].mean()), 3),
    "share_rate": round(float(interactions_df["is_share"].mean()), 3),
    "positive_rate": round(float(interactions_df["watch_label"].mean()), 3),
}

summary_path = OUTPUT_DIR / "interaction_summary.json"
write_json(summary, summary_path)

print("saved:", interactions_path_jsonl)
print("saved:", interactions_path_parquet)
print("saved:", summary_path)
print(json.dumps(summary, indent=2, ensure_ascii=False))
interactions_df.head(5)


In [ ]:
positive_df = interactions_df[interactions_df["watch_label"] == 1].copy()
positive_df = positive_df.sort_values(["user_id", "event_time", "position_in_session"]).reset_index(drop=True)

train_rows = []
valid_rows = []
test_rows = []

for user_id, group in positive_df.groupby("user_id"):
    group = group.reset_index(drop=True)
    item_ids = group["item_id"].astype(int).tolist()
    clip_ids = group["clip_id"].tolist()
    times = group["event_time"].tolist()
    sessions = group["session_id"].tolist()

    if len(item_ids) < 2:
        continue

    train_stop = len(item_ids) - 2 if len(item_ids) >= 4 else len(item_ids) - 1
    train_stop = max(train_stop, 1)

    for target_idx in range(1, train_stop):
        start_idx = max(0, target_idx - MAX_SEQ_LEN)
        seq_item_ids = item_ids[start_idx:target_idx]
        if not seq_item_ids:
            continue
        train_rows.append({
            "sample_id": f"train_{len(train_rows) + 1:09d}",
            "user_id": user_id,
            "session_id": sessions[target_idx],
            "input_item_ids": seq_item_ids,
            "input_clip_ids": clip_ids[start_idx:target_idx],
            "target_item_id": int(item_ids[target_idx]),
            "target_clip_id": clip_ids[target_idx],
            "sequence_timestamps": times[start_idx:target_idx],
            "target_timestamp": times[target_idx],
            "target_label": 1,
        })

    if len(item_ids) >= 3:
        valid_target_idx = len(item_ids) - 2
        valid_start_idx = max(0, valid_target_idx - MAX_SEQ_LEN)
        valid_rows.append({
            "sample_id": f"valid_{len(valid_rows) + 1:09d}",
            "user_id": user_id,
            "session_id": sessions[valid_target_idx],
            "input_item_ids": item_ids[valid_start_idx:valid_target_idx],
            "input_clip_ids": clip_ids[valid_start_idx:valid_target_idx],
            "target_item_id": int(item_ids[valid_target_idx]),
            "target_clip_id": clip_ids[valid_target_idx],
            "sequence_timestamps": times[valid_start_idx:valid_target_idx],
            "target_timestamp": times[valid_target_idx],
            "target_label": 1,
        })

        test_target_idx = len(item_ids) - 1
        test_start_idx = max(0, test_target_idx - MAX_SEQ_LEN)
        test_rows.append({
            "sample_id": f"test_{len(test_rows) + 1:09d}",
            "user_id": user_id,
            "session_id": sessions[test_target_idx],
            "input_item_ids": item_ids[test_start_idx:test_target_idx],
            "input_clip_ids": clip_ids[test_start_idx:test_target_idx],
            "target_item_id": int(item_ids[test_target_idx]),
            "target_clip_id": clip_ids[test_target_idx],
            "sequence_timestamps": times[test_start_idx:test_target_idx],
            "target_timestamp": times[test_target_idx],
            "target_label": 1,
        })

train_path = OUTPUT_DIR / "sasrec_train_sequences.jsonl"
valid_path = OUTPUT_DIR / "sasrec_valid_sequences.jsonl"
test_path = OUTPUT_DIR / "sasrec_test_sequences.jsonl"

write_jsonl(train_rows, train_path)
write_jsonl(valid_rows, valid_path)
write_jsonl(test_rows, test_path)

print("positive_events:", len(positive_df))
print("train_samples:", len(train_rows))
print("valid_samples:", len(valid_rows))
print("test_samples:", len(test_rows))
print("saved:", train_path)
print("saved:", valid_path)
print("saved:", test_path)
pd.DataFrame(train_rows).head(5)
